# Swarm Seek: your own fitness function

This notebook shows how to optimize a **user-defined objective** with Swarm Seek:
you supply the fitness function; the colony proposes parameter vectors and
consumes the scores you return. Use it when built-in benchmarks are not your
problem—simulators, residuals, losses, or any black-box score over box bounds.

*Dual purpose:* the narrative follows the scientific tutorial arc below; a
seeded ask/tell run also asserts a numeric pass criterion for CI (`nbmake`).

## Setup

Install once with `pip install -e ".[dev]"`. This notebook was written against
the package version printed below.

In [1]:
from __future__ import annotations

import numpy as np

from swarm_seek import ABC, ContinuousSpace, __version__

print(f"swarm_seek {__version__}")

swarm_seek 0.1.0a0


## Motivation and background

Built-in objectives (`sphere`, `rastrigin`, …) are for testing and demos. Real
work usually means a **parameters → score** map you already own, for example:

- residual of a model versus measured data
- cost from a simulator
- negative log-likelihood or validation loss

Without an ask/tell (or `minimize`) API you would hand-roll a search loop or
glue a general optimizer to your evaluator. Swarm Seek only needs:

1. A `ContinuousSpace` (box bounds per parameter).
2. Fitness values for each candidate row it asks you to evaluate.

**Contract:** candidates arrive as a NumPy array of shape `(n_ask, n_dim)`.
Return a length-`n_ask` vector of finite floats. Lower is better when
`sense="minimize"` (default).

## Minimal example

Stand-in objective: a **synthetic** 2-D residual with known minimizer
`(1.5, -0.5)`. It is not a real calibration against data—only a clear target so
you can see recovery. Swap `my_fitness` for your metric; keep the array shapes.

In [2]:
def my_fitness(x: np.ndarray) -> np.ndarray:
    """Synthetic residual with known minimizer at (1.5, -0.5).

    Parameters
    ----------
    x
        Array of shape ``(n_candidates, n_dim)`` or ``(n_dim,)``.

    Returns
    -------
    np.ndarray
        Fitness per candidate, shape ``(n_candidates,)`` (or scalar for 1-D ``x``).
    """
    x = np.asarray(x, dtype=np.float64)
    return (x[..., 0] - 1.5) ** 2 + (x[..., 1] + 0.5) ** 2


BOUNDS = [(-5.0, 5.0), (-5.0, 5.0)]
space = ContinuousSpace(BOUNDS)

# Smallest useful run: in-process minimize
colony_min = ABC(
    space,
    variant="original",
    pop_size=20,
    limit=100,
    max_evals=3_000,
    seed=0,
)
best_min = colony_min.minimize(my_fitness)
print(
    f"minimize → x={best_min.x}, fitness={best_min.fitness:.6e}, "
    f"n_evals={colony_min.n_evals}"
)

minimize → x=[ 1.5 -0.5], fitness=4.213424e-25, n_evals=3005


## Progressive deep dive

### Shape check

Confirm your callable accepts a batch and returns one score per row.

In [3]:
batch = np.array([[0.0, 0.0], [1.5, -0.5], [2.0, 1.0]], dtype=np.float64)
scores = my_fitness(batch)
assert scores.shape == (3,)
assert np.isfinite(scores).all()
print("batch scores (expect ~[2.5, 0, 2.5]):", scores)

batch scores (expect ~[2.5, 0, 2.5]): [2.5 0.  2.5]


### Ask/tell for external evaluation

`minimize` is convenience when the objective is a fast in-process callable.
Prefer **ask/tell** when evaluation is external, expensive, or parallel (cluster
jobs, another process, instrument). Empty batches (`candidates.size == 0`) can
occur in some phases—call `tell([])` and continue.

**Verification:** seeded ask/tell best fitness must be below `1e-3`.

In [4]:
SEED = 1
MAX_EVALS = 3_000
PASS_THRESHOLD = 1e-3

colony = ABC(
    ContinuousSpace(BOUNDS),
    variant="gabc",
    pop_size=20,
    limit=100,
    max_evals=MAX_EVALS,
    seed=SEED,
)

while not colony.converged:
    candidates = colony.ask()
    if candidates.size == 0:
        colony.tell([])
        continue
    # --- your evaluation goes here (batch or one-by-one) ---
    fitnesses = np.asarray(my_fitness(candidates), dtype=np.float64)
    colony.tell(fitnesses)

best = colony.best
print(
    f"ask/tell → x={best.x}, fitness={best.fitness:.6e}, "
    f"n_evals={colony.n_evals}, n_iters={colony.n_iters}"
)
assert best.fitness < PASS_THRESHOLD, (
    f"Pass criterion failed: best fitness {best.fitness!r} >= {PASS_THRESHOLD}"
)
print("PASS: custom objective ask/tell below threshold")

ask/tell → x=[ 1.5 -0.5], fitness=0.000000e+00, n_evals=3008, n_iters=95
PASS: custom objective ask/tell below threshold


### Wiring tips

- **Bounds:** put physical or policy limits in `ContinuousSpace`; out-of-box
  proposals are clipped (repair).
- **Batch vs scalar:** prefer vectorized `(n, d) → (n,)`; for scalar-only APIs,
  loop over rows and stack scores.
- **Maximize:** `sense="maximize"` when larger scores are better.
- **Budgets:** `max_evals`, `max_iters`, and/or `stall_evals`.
- **Reproducibility:** set `seed=` for deterministic candidate streams.
- **Variants:** try `"gabc"`, `"qabc"`, or `"mabc"` with the same objective
  (`examples/02_variant_compare.ipynb`).

## Results

On this synthetic residual the known minimizer is `(1.5, -0.5)` with fitness
`0`. Look at the printed `x` and `fitness` from the cells above:

- **`minimize` (seed 0):** recovered coordinates should sit on (or extremely
  near) the target; fitness should be many orders of magnitude below `1e-3`.
- **Ask/tell (seed 1, GABC):** same target recovery; the assert requires
  `fitness < 1e-3` and prints `PASS` when CI criteria hold.
- **`n_evals`:** slightly above the configured budget is normal—the colony
  finishes the pending ask batch that crosses the limit.

There is no figure here; the numeric recovery against a known optimum is the
evidence that a user-defined callable plugged in correctly.

## Interpretation / discussion

The Motivation case was “I already have a score function.” Both APIs solve that:

- Use **`minimize`** when evaluation is a normal Python callable in the same
  process—less boilerplate.
- Use **ask/tell** when you must own scheduling, I/O, or parallelism; Swarm Seek
  never calls your simulator itself.

Tradeoffs and breakdowns:

- This demo is **smooth and noise-free**. Noisy, discontinuous, or
  multi-objective scores need more budget, different stopping rules, or another
  method—ABC here is single-objective continuous search in a box.
- **Discrete or mixed** variables are out of scope for `0.1.x`
  (`ContinuousSpace` only).
- Constraints beyond box bounds are not enforced except by encoding them in
  the fitness (penalties) or tightening bounds.
- Very expensive evaluations: prefer ask/tell and evaluate batches in parallel
  outside the colony.

## Takeaways

- **When to use:** continuous parameters in a box; you already have a scalar
  fitness (or can batch one); you want citation-linked ABC with ask/tell.
- **When not to:** multi-objective Pareto search, discrete/mixed spaces, or
  problems that need gradients / local QP—use a tool built for those.
- **Gotchas:** return finite `(n_ask,)` scores; handle empty asks with
  `tell([])`; set `seed` for reproducibility; match `sense` to “lower/higher
  is better.”

## Further reading

- Quickstart (ask/tell sketch):
  https://swarm-seek.readthedocs.io/en/latest/quickstart.html
- API reference:
  https://swarm-seek.readthedocs.io/en/latest/api.html
- Related notebooks in this repo:
  `examples/01_ask_tell_sphere.ipynb`,
  `examples/02_variant_compare.ipynb`
- Artificial Bee Colony background (optional):
  Karaboga & Basturk (2007),
  https://doi.org/10.1007/s10898-007-9149-x